<a href="https://colab.research.google.com/github/gabriela0402/pathfolio-AI/blob/main/notebooks/00_environment_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pathfolio — Verificação e ingestão inicial

Este notebook será usado para testar a leitura de textos digitados,
arquivos PDF e arquivos DOCX.


## 1. Preparação do ambiente

Nesta etapa, o Pathfolio receberá informações escritas manualmente e as transformará em uma estrutura padronizada.


In [ ]:
from datetime import datetime, timezone

print("Ambiente do Pathfolio funcionando!")


Ambiente do Pathfolio funcionando!


In [ ]:
def create_text_source(
    content: str,
    source_name: str,
    category: str = "manual_text"
) -> dict:
    """
    Converte um texto digitado pelo usuário em uma fonte padronizada.
    """

    cleaned_content = content.strip()

    if not cleaned_content:
        raise ValueError("O conteúdo não pode estar vazio.")

    return {
        "source_type": "manual_text",
        "source_name": source_name,
        "category": category,
        "content": cleaned_content,
        "metadata": {
            "created_at": datetime.now(timezone.utc).isoformat(),
            "page": None,
            "file_name": None
        }
    }


In [ ]:
manual_source = create_text_source(
    content="""
    Participei de uma iniciação científica envolvendo a comparação de modelos
    de inteligência artificial aplicados à correção de redações. Trabalhei com
    uma base de dados extensa, análise de resultados e comparação de métricas.
    """,
    source_name="Experiência acadêmica",
    category="academic_experience"
)

manual_source


{'source_type': 'manual_text',
 'source_name': 'Experiência acadêmica',
 'category': 'academic_experience',
 'content': 'Participei de uma iniciação científica envolvendo a comparação de modelos\n    de inteligência artificial aplicados à correção de redações. Trabalhei com\n    uma base de dados extensa, análise de resultados e comparação de métricas.',
 'metadata': {'created_at': '2026-11-06T22:12:16.206045+00:00',
  'page': None,
  'file_name': None}}

In [ ]:
assert manual_source["source_type"] == "manual_text"
assert manual_source["source_name"] == "Experiência acadêmica"
assert manual_source["category"] == "academic_experience"
assert len(manual_source["content"]) > 0
assert manual_source["metadata"]["page"] is None

print("Teste de texto manual aprovado.")


Teste de texto manual aprovado.


In [ ]:
try:
    create_text_source(
        content="   ",
        source_name="Texto vazio"
    )
except ValueError as error:
    print(f"Validação funcionando: {error}")


Validação funcionando: O conteúdo não pode estar vazio.


## 2. Leitura de documentos

Nesta etapa, o Pathfolio será capaz de ler arquivos PDF e DOCX.

Cada documento será convertido para uma ou mais fontes padronizadas,
preservando o nome do arquivo e, quando possível, a página de origem.


In [ ]:
!pip -q install pypdf python-docx


In [ ]:
from pathlib import Path

from pypdf import PdfReader
from docx import Document
from google.colab import files


In [ ]:
def read_pdf(file_path: str) -> list[dict]:
    """
    Extrai o texto de um PDF página por página.
    """

    reader = PdfReader(file_path)
    sources = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()

        if not text:
            continue

        sources.append({
            "source_type": "pdf",
            "source_name": Path(file_path).name,
            "category": "uploaded_document",
            "content": text,
            "metadata": {
                "created_at": datetime.now(timezone.utc).isoformat(),
                "page": page_number,
                "file_name": Path(file_path).name
            }
        })

    return sources


In [ ]:
def read_docx(file_path: str) -> list[dict]:
    """
    Extrai o texto dos parágrafos de um arquivo DOCX.
    """

    document = Document(file_path)
    paragraphs = []

    for paragraph in document.paragraphs:
        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    content = "\n".join(paragraphs).strip()

    if not content:
        return []

    return [{
        "source_type": "docx",
        "source_name": Path(file_path).name,
        "category": "uploaded_document",
        "content": content,
        "metadata": {
            "created_at": datetime.now(timezone.utc).isoformat(),
            "page": None,
            "file_name": Path(file_path).name
        }
    }]


In [ ]:
def read_document(file_path: str) -> list[dict]:
    """
    Lê um PDF ou DOCX usando o leitor correspondente.
    """

    extension = Path(file_path).suffix.lower()

    if extension == ".pdf":
        return read_pdf(file_path)

    if extension == ".docx":
        return read_docx(file_path)

    raise ValueError(
        f"Formato não suportado: {extension}. "
        "Envie um arquivo PDF ou DOCX."
    )


In [ ]:
uploaded_files = files.upload()


Saving 03_descricao_projeto_ficticio.docx to 03_descricao_projeto_ficticio.docx
Saving 02_relatorio_projeto_ficticio.pdf to 02_relatorio_projeto_ficticio.pdf
Saving 01_curriculo_candidato_ficticio.pdf to 01_curriculo_candidato_ficticio.pdf


In [ ]:
uploaded_sources = []

for file_name in uploaded_files.keys():
    document_sources = read_document(file_name)
    uploaded_sources.extend(document_sources)

print(f"Fontes extraídas: {len(uploaded_sources)}")

for source in uploaded_sources:
    print(f"Arquivo: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Página: {source['metadata']['page']}")
    print(f"Caracteres: {len(source['content'])}")
    print("-" * 50)


Fontes extraídas: 4
Arquivo: 03_descricao_projeto_ficticio.docx
Tipo: docx
Página: None
Caracteres: 807
--------------------------------------------------
Arquivo: 02_relatorio_projeto_ficticio.pdf
Tipo: pdf
Página: 1
Caracteres: 870
--------------------------------------------------
Arquivo: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Página: 1
Caracteres: 907
--------------------------------------------------
Arquivo: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Página: 2
Caracteres: 522
--------------------------------------------------


In [ ]:
all_sources = [manual_source] + uploaded_sources

print(f"Total de fontes: {len(all_sources)}")

for index, source in enumerate(all_sources, start=1):
    print(
        f"{index}. "
        f"{source['source_name']} | "
        f"{source['source_type']} | "
        f"{len(source['content'])} caracteres"
    )


Total de fontes: 5
1. Experiência acadêmica | manual_text | 231 caracteres
2. 03_descricao_projeto_ficticio.docx | docx | 807 caracteres
3. 02_relatorio_projeto_ficticio.pdf | pdf | 870 caracteres
4. 01_curriculo_candidato_ficticio.pdf | pdf | 907 caracteres
5. 01_curriculo_candidato_ficticio.pdf | pdf | 522 caracteres


In [ ]:
for source in all_sources:
    preview = source["content"][:300].replace("\n", " ")

    print(f"Fonte: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Prévia: {preview}")
    print("=" * 80)


Fonte: Experiência acadêmica
Tipo: manual_text
Prévia: Participei de uma iniciação científica envolvendo a comparação de modelos     de inteligência artificial aplicados à correção de redações. Trabalhei com     uma base de dados extensa, análise de resultados e comparação de métricas.
Fonte: 03_descricao_projeto_ficticio.docx
Tipo: docx
Prévia: Descrição de Projeto — Documento de Teste Projeto: Painel de acompanhamento de estudos Contexto Aplicação web fictícia criada para organizar metas de estudo, registrar sessões e sugerir revisões com base no histórico do usuário. Contribuições realizadas Levantamento dos requisitos principais da apli
Fonte: 02_relatorio_projeto_ficticio.pdf
Tipo: pdf
Prévia: Relatório de Projeto — Documento de Teste Projeto: Classificação de feedbacks acadêmicos Responsável: Marina Oliveira Resumo O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks acadêmicos por tema e prioridade, facilitando a análise por professores. A

In [ ]:
assert len(all_sources) >= 1

for source in all_sources:
    assert source["content"].strip()
    assert source["source_name"]
    assert source["source_type"] in {"manual_text", "pdf", "docx"}
    assert "metadata" in source
    assert "file_name" in source["metadata"]

print("Testes de ingestão de documentos aprovados.")


Testes de ingestão de documentos aprovados.


In [ ]:
try:
    read_document("arquivo.txt")
except ValueError as error:
    print(f"Validação funcionando: {error}")


Validação funcionando: Formato não suportado: .txt. Envie um arquivo PDF ou DOCX.


## 3. Limpeza e divisão dos conteúdos

Nesta etapa, os conteúdos extraídos de textos manuais, PDFs e DOCX serão limpos e divididos em trechos menores.

Esses trechos serão utilizados posteriormente para criar embeddings e realizar busca semântica.


In [ ]:
import re


def clean_text(text: str) -> str:
    """
    Limpa espaços e quebras de linha desnecessárias.
    """

    if not isinstance(text, str):
        raise TypeError("O conteúdo precisa ser uma string.")

    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


In [ ]:
dirty_text = """
    Python     e análise de dados.


    Experiência com modelos de linguagem.
"""

cleaned_text = clean_text(dirty_text)

print(cleaned_text)


Python e análise de dados.

 Experiência com modelos de linguagem.


In [ ]:
assert "     " not in cleaned_text
assert cleaned_text.strip() == cleaned_text

print("Teste de limpeza aprovado.")


Teste de limpeza aprovado.


In [ ]:
cleaned_sources = []

for source in all_sources:
    cleaned_source = source.copy()
    cleaned_source["content"] = clean_text(source["content"])
    cleaned_sources.append(cleaned_source)

print(f"Fontes processadas: {len(cleaned_sources)}")


Fontes processadas: 5


In [ ]:
for source in cleaned_sources:
    print(f"Fonte: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Caracteres: {len(source['content'])}")
    print(f"Prévia: {source['content'][:200]}")
    print("=" * 80)


Fonte: Experiência acadêmica
Tipo: manual_text
Caracteres: 225
Prévia: Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com
 uma base de dados extensa, análise de resultados 
Fonte: 03_descricao_projeto_ficticio.docx
Tipo: docx
Caracteres: 807
Prévia: Descrição de Projeto — Documento de Teste
Projeto: Painel de acompanhamento de estudos
Contexto
Aplicação web fictícia criada para organizar metas de estudo, registrar sessões e sugerir revisões com b
Fonte: 02_relatorio_projeto_ficticio.pdf
Tipo: pdf
Caracteres: 870
Prévia: Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técnicas de processamento de linguagem natural para or
Fonte: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Caracteres: 907
Prévia: Currículo de Demonstração
Marina Oliveira
Candidata fictícia para testes do Pathfolio
Objetivo
Bu

In [ ]:
def split_text(
    text: str,
    chunk_size: int = 800,
    chunk_overlap: int = 120
) -> list[str]:
    """
    Divide um texto em trechos com sobreposição.

    A sobreposição ajuda a preservar o contexto entre dois trechos.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size precisa ser maior que zero.")

    if chunk_overlap < 0:
        raise ValueError("chunk_overlap não pode ser negativo.")

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap precisa ser menor que chunk_size."
        )

    text = clean_text(text)

    if not text:
        return []

    chunks = []
    start = 0
    text_length = len(text)
    step = chunk_size - chunk_overlap

    while start < text_length:
        end = min(start + chunk_size, text_length)
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        start += step

    return chunks


In [ ]:
test_text = (
    "Python é uma linguagem utilizada em análise de dados. "
    "Machine Learning permite construir modelos preditivos. "
    "Modelos de linguagem podem processar textos e gerar respostas. "
    "A avaliação dos resultados é importante para entender limitações."
)

test_chunks = split_text(
    test_text,
    chunk_size=100,
    chunk_overlap=20
)

for index, chunk in enumerate(test_chunks):
    print(f"Trecho {index}:")
    print(chunk)
    print("-" * 50)


Trecho 0:
Python é uma linguagem utilizada em análise de dados. Machine Learning permite construir modelos pre
--------------------------------------------------
Trecho 1:
onstruir modelos preditivos. Modelos de linguagem podem processar textos e gerar respostas. A avalia
--------------------------------------------------
Trecho 2:
respostas. A avaliação dos resultados é importante para entender limitações.
--------------------------------------------------


In [ ]:
assert len(test_chunks) > 1
assert all(len(chunk) > 0 for chunk in test_chunks)

print("Teste de divisão de texto aprovado.")


Teste de divisão de texto aprovado.


In [ ]:
def create_chunks(
    sources: list[dict],
    chunk_size: int = 800,
    chunk_overlap: int = 120
) -> list[dict]:
    """
    Divide todas as fontes em trechos e preserva seus metadados.
    """

    chunks = []
    chunk_counter = 0

    for source in sources:
        source_chunks = split_text(
            source["content"],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        for chunk_index, content in enumerate(source_chunks):
            chunks.append({
                "chunk_id": f"chunk-{chunk_counter:04d}",
                "content": content,
                "source_name": source["source_name"],
                "source_type": source["source_type"],
                "category": source["category"],
                "metadata": {
                    "page": source["metadata"].get("page"),
                    "file_name": source["metadata"].get("file_name"),
                    "chunk_index": chunk_index,
                    "source_created_at": source["metadata"].get(
                        "created_at"
                    )
                }
            })

            chunk_counter += 1

    return chunks


In [ ]:
chunks = create_chunks(cleaned_sources)

print(f"Total de chunks criados: {len(chunks)}")


Total de chunks criados: 8


In [ ]:
for chunk in chunks[:5]:
    print(f"ID: {chunk['chunk_id']}")
    print(f"Fonte: {chunk['source_name']}")
    print(f"Tipo: {chunk['source_type']}")
    print(f"Página: {chunk['metadata']['page']}")
    print(f"Índice: {chunk['metadata']['chunk_index']}")
    print(f"Conteúdo: {chunk['content'][:300]}")
    print("=" * 80)


ID: chunk-0000
Fonte: Experiência acadêmica
Tipo: manual_text
Página: None
Índice: 0
Conteúdo: Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com
 uma base de dados extensa, análise de resultados e comparação de métricas.
ID: chunk-0001
Fonte: 03_descricao_projeto_ficticio.docx
Tipo: docx
Página: None
Índice: 0
Conteúdo: Descrição de Projeto — Documento de Teste
Projeto: Painel de acompanhamento de estudos
Contexto
Aplicação web fictícia criada para organizar metas de estudo, registrar sessões e sugerir revisões com base no histórico do usuário.
Contribuições realizadas
Levantamento dos requisitos principais da apli
ID: chunk-0002
Fonte: 03_descricao_projeto_ficticio.docx
Tipo: docx
Página: None
Índice: 1
Conteúdo: ia do usuário.
Aprendizados
O projeto ajudou a transformar uma necessidade de usuário em funcionalidades menores e organizadas.
ID: chunk-0003
Fonte: 02_relatorio_projeto_ficti

In [ ]:
assert len(chunks) > 0

for chunk in chunks:
    assert chunk["chunk_id"]
    assert chunk["content"]
    assert chunk["source_name"]
    assert chunk["source_type"] in {"manual_text", "pdf", "docx"}
    assert "metadata" in chunk
    assert "page" in chunk["metadata"]

print("Testes de chunks e metadados aprovados.")


Testes de chunks e metadados aprovados.


In [ ]:
from collections import Counter

chunks_per_source = Counter(
    chunk["source_name"]
    for chunk in chunks
)

for source_name, quantity in chunks_per_source.items():
    print(f"{source_name}: {quantity} chunks")


Experiência acadêmica: 1 chunks
03_descricao_projeto_ficticio.docx: 2 chunks
02_relatorio_projeto_ficticio.pdf: 2 chunks
01_curriculo_candidato_ficticio.pdf: 3 chunks


In [ ]:
def keyword_search(
    chunks: list[dict],
    query: str
) -> list[dict]:
    """
    Retorna chunks que contêm todas as palavras relevantes da busca.
    """

    query_terms = [
        term.lower()
        for term in re.findall(r"\w+", query)
        if len(term) > 2
    ]

    results = []

    for chunk in chunks:
        content = chunk["content"].lower()

        if all(term in content for term in query_terms):
            results.append(chunk)

    return results


In [ ]:
results = keyword_search(
    chunks,
    "Python análise de dados"
)

print(f"Resultados encontrados: {len(results)}")

for result in results:
    print(result["source_name"])
    print(result["content"])
    print("-" * 80)


Resultados encontrados: 2
02_relatorio_projeto_ficticio.pdf
Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks
acadêmicos por tema e prioridade, facilitando a análise por professores.
Atividades realizadas

Coleta e limpeza de exemplos textuais fictícios.

Definição de categorias para classificação dos feedbacks.

Criação de uma rotina de pré-processamento em Python.

Avaliação qualitativa dos resultados e análise de erros.

Registro das decisões e limitações do experimento.
Tecnologias e conceitos
Python, Pandas, classificação de texto, análise exploratória de dados, métricas de avaliação e Git.
Resultado
O experimento identificou situações em que textos ambíguos exigiam revisão
--------------------------------------------------------------------------------
01_curriculo_candidato_ficticio.pdf
Currículo de Demonstra

In [ ]:
import json

chunks_output_path = "pathfolio_chunks_debug.json"

with open(chunks_output_path, "w", encoding="utf-8") as file:
    json.dump(chunks, file, ensure_ascii=False, indent=2)

print(f"Arquivo salvo: {chunks_output_path}")


Arquivo salvo: pathfolio_chunks_debug.json


## 4. Embeddings e busca semântica

Nesta etapa, cada chunk será convertido em um vetor numérico chamado embedding.

Chunks semanticamente parecidos ficarão próximos no espaço vetorial. Assim,
o Pathfolio poderá encontrar evidências relacionadas a uma vaga mesmo quando
as palavras utilizadas forem diferentes.


In [ ]:
!pip -q install sentence-transformers scikit-learn


In [ ]:
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

print(f"Modelo carregado: {embedding_model_name}")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo carregado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [ ]:
chunk_texts = [
    chunk["content"]
    for chunk in chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f"Quantidade de embeddings: {len(chunk_embeddings)}")
print(f"Dimensão de cada embedding: {chunk_embeddings.shape[1]}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Quantidade de embeddings: 8
Dimensão de cada embedding: 384


In [ ]:
assert len(chunk_embeddings) == len(chunks)
assert chunk_embeddings.ndim == 2
assert chunk_embeddings.shape[1] > 0
assert np.isfinite(chunk_embeddings).all()

print("Teste de embeddings aprovado.")


Teste de embeddings aprovado.


In [ ]:
embedding_norms = np.linalg.norm(chunk_embeddings, axis=1)

print(embedding_norms[:5])


[1.         0.99999994 1.         0.99999994 1.        ]


In [ ]:
def semantic_search(
    query: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 5
) -> list[dict]:
    """
    Retorna os chunks semanticamente mais próximos da consulta.
    """

    if not query or not query.strip():
        raise ValueError("A consulta não pode estar vazia.")

    if len(chunks) != len(chunk_embeddings):
        raise ValueError(
            "A quantidade de chunks precisa ser igual à quantidade de embeddings."
        )

    if top_k <= 0:
        raise ValueError("top_k precisa ser maior que zero.")

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    ranked_indices = np.argsort(similarities)[::-1]
    ranked_indices = ranked_indices[:min(top_k, len(chunks))]

    results = []

    for rank, index in enumerate(ranked_indices, start=1):
        result = chunks[index].copy()
        result["similarity_score"] = float(similarities[index])
        result["rank"] = rank
        results.append(result)

    return results


In [ ]:
query = """
Experiência com Python, análise de dados e avaliação de modelos de inteligência artificial.
"""

semantic_results = semantic_search(
    query=query,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)


In [ ]:
for result in semantic_results:
    print(f"Rank: {result['rank']}")
    print(f"Similaridade: {result['similarity_score']:.4f}")
    print(f"Fonte: {result['source_name']}")
    print(f"Tipo: {result['source_type']}")
    print(f"Página: {result['metadata']['page']}")
    print(f"Conteúdo: {result['content']}")
    print("=" * 100)


Rank: 1
Similaridade: 0.6570
Fonte: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Página: 2
Conteúdo: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento de produtos digitais.
Rank: 2
Similaridade: 0.6483
Fonte: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Página: 1
Conteúdo: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
Rank: 3
Similaridade: 0.6235
Fonte: Experiênc

In [ ]:
query_semantic = """
Vivência em experimentos com modelos computacionais capazes de trabalhar com linguagem humana.
"""

semantic_results_2 = semantic_search(
    query=query_semantic,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)

for result in semantic_results_2:
    print(
        f"Rank {result['rank']} | "
        f"Score {result['similarity_score']:.4f} | "
        f"{result['source_name']}"
    )
    print(result["content"][:500])
    print("-" * 80)


Rank 1 | Score 0.6798 | 01_curriculo_candidato_ficticio.pdf
s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
--------------------------------------------------------------------------------
Rank 2 | Score 0.6141 | 01_curriculo_candidato_ficticio.pdf
Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento
----------------------------------------------------------------------

In [ ]:
job_description = """
Estágio em Inteligência Artificial.

A pessoa candidata irá apoiar projetos de análise de dados e desenvolvimento
de soluções baseadas em modelos de linguagem. É desejável conhecimento em
Python, machine learning, processamento de linguagem natural, avaliação de
modelos e documentação de experimentos.
"""


In [ ]:
job_results = semantic_search(
    query=job_description,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)

for result in job_results:
    print(f"Rank: {result['rank']}")
    print(f"Score: {result['similarity_score']:.4f}")
    print(f"Fonte: {result['source_name']}")
    print(f"Página: {result['metadata']['page']}")
    print(f"Trecho: {result['content']}")
    print("=" * 100)


Rank: 1
Score: 0.7510
Fonte: 01_curriculo_candidato_ficticio.pdf
Página: 2
Trecho: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento de produtos digitais.
Rank: 2
Score: 0.7422
Fonte: 01_curriculo_candidato_ficticio.pdf
Página: 1
Trecho: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
Rank: 3
Score: 0.7071
Fonte: 01_curriculo_candidato_ficticio.pdf
Página: 1
Trecho: 

In [ ]:
def format_evidence(results: list[dict]) -> list[dict]:
    """
    Formata os resultados para serem apresentados como evidências.
    """

    evidences = []

    for result in results:
        evidences.append({
            "evidence_id": result["chunk_id"],
            "rank": result["rank"],
            "similarity_score": result["similarity_score"],
            "source_name": result["source_name"],
            "source_type": result["source_type"],
            "page": result["metadata"].get("page"),
            "content": result["content"]
        })

    return evidences


In [ ]:
evidences = format_evidence(job_results)

for evidence in evidences:
    print(evidence)


{'evidence_id': 'chunk-0007', 'rank': 1, 'similarity_score': 0.7510434985160828, 'source_name': '01_curriculo_candidato_ficticio.pdf', 'source_type': 'pdf', 'page': 2, 'content': 'Currículo de Demonstração — continuação\nHabilidades\nPython · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de\nDados · Git · GitHub · SQL básico\nCursos e atividades\n\x7f\nIntrodução a Machine Learning — curso livre.\n\x7f\nPython para análise de dados — curso livre.\n\x7f\nParticipação em grupo de estudos sobre Inteligência Artificial Generativa.\nInformações adicionais\nInteresse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e\ndesenvolvimento de produtos digitais.'}
{'evidence_id': 'chunk-0006', 'rank': 2, 'similarity_score': 0.7422292828559875, 'source_name': '01_curriculo_candidato_ficticio.pdf', 'source_type': 'pdf', 'page': 1, 'content': 's\nAssistente de estudos com processamento de linguagem natural\nProjeto experimental para ge

In [ ]:
def filter_by_similarity(
    results: list[dict],
    minimum_score: float = 0.35
) -> list[dict]:
    """
    Mantém somente resultados acima da similaridade mínima.
    """

    return [
        result
        for result in results
        if result["similarity_score"] >= minimum_score
    ]


In [ ]:
filtered_results = filter_by_similarity(
    job_results,
    minimum_score=0.35
)

print(f"Resultados antes do filtro: {len(job_results)}")
print(f"Resultados depois do filtro: {len(filtered_results)}")


Resultados antes do filtro: 5
Resultados depois do filtro: 5


In [ ]:
unrelated_results = semantic_search(
    query="Experiência profissional em engenharia mecânica e manutenção industrial.",
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)

for result in unrelated_results:
    print(
        f"{result['similarity_score']:.4f} | "
        f"{result['source_name']} | "
        f"{result['content'][:150]}"
    )


0.2714 | 03_descricao_projeto_ficticio.docx | Descrição de Projeto — Documento de Teste
Projeto: Painel de acompanhamento de estudos
Contexto
Aplicação web fictícia criada para organizar metas de 
0.2398 | 01_curriculo_candidato_ficticio.pdf | Currículo de Demonstração
Marina Oliveira
Candidata fictícia para testes do Pathfolio
Objetivo
Buscar oportunidade de estágio em Inteligência Artifici
0.2091 | 02_relatorio_projeto_ficticio.pdf | Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técn
0.2079 | Experiência acadêmica | Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com

0.2036 | 01_curriculo_candidato_ficticio.pdf | s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de


In [ ]:
import json

semantic_results_path = "pathfolio_semantic_search_debug.json"

with open(semantic_results_path, "w", encoding="utf-8") as file:
    json.dump(
        job_results,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Arquivo salvo: {semantic_results_path}")


Arquivo salvo: pathfolio_semantic_search_debug.json


In [ ]:
assert len(job_results) > 0
assert len(job_results) <= 5

for index, result in enumerate(job_results, start=1):
    assert result["rank"] == index
    assert 0 <= result["similarity_score"] <= 1
    assert result["content"]
    assert result["source_name"]

for first, second in zip(job_results, job_results[1:]):
    assert first["similarity_score"] >= second["similarity_score"]

print("Testes de busca semântica aprovados.")


Testes de busca semântica aprovados.


## 5. Análise de competências e lacunas

Nesta etapa, o Pathfolio analisará uma descrição de vaga e verificará
quais competências possuem evidências nas informações fornecidas pelo candidato.

A aplicação não deverá considerar uma competência como comprovada apenas
porque ela aparece na descrição da vaga. Ela precisará estar relacionada
a algum trecho recuperado das fontes do candidato.


In [ ]:
job_description = """
Estágio em Inteligência Artificial

A pessoa candidata irá apoiar projetos de análise de dados e desenvolvimento
de soluções baseadas em modelos de linguagem.

Requisitos:
- Conhecimento em Python;
- Noções de machine learning;
- Interesse em modelos de linguagem;
- Capacidade de analisar dados;
- Boa comunicação para apresentar resultados.

Diferenciais:
- Experiência com processamento de linguagem natural;
- Experiência com avaliação de modelos;
- Participação em projetos de pesquisa.
"""


In [ ]:
skill_catalog = [
    "Python",
    "Machine Learning",
    "Análise de dados",
    "Modelos de linguagem",
    "Processamento de Linguagem Natural",
    "Avaliação de modelos",
    "Pesquisa",
    "Git",
    "GitHub",
    "SQL",
    "React",
    "JavaScript",
    "Comunicação"
]


In [ ]:
skill_aliases = {
    "Python": [
        "python"
    ],
    "Machine Learning": [
        "machine learning",
        "aprendizado de máquina",
        "ml"
    ],
    "Análise de dados": [
        "análise de dados",
        "analise de dados",
        "análise dos dados",
        "analise dos dados",
        "data analysis"
    ],
    "Modelos de linguagem": [
        "modelos de linguagem",
        "modelo de linguagem",
        "large language model",
        "large language models",
        "llm",
        "llms"
    ],
    "Processamento de Linguagem Natural": [
        "processamento de linguagem natural",
        "natural language processing",
        "nlp"
    ],
    "Avaliação de modelos": [
        "avaliação de modelos",
        "avaliacao de modelos",
        "avaliação de modelagem",
        "model evaluation"
    ],
    "Pesquisa": [
        "pesquisa",
        "iniciação científica",
        "iniciacao cientifica",
        "investigação",
        "investigacao"
    ],
    "Git": [
        "git"
    ],
    "GitHub": [
        "github"
    ],
    "SQL": [
        "sql"
    ],
    "React": [
        "react"
    ],
    "JavaScript": [
        "javascript",
        "java script"
    ],
    "Comunicação": [
        "comunicação",
        "comunicacao",
        "apresentação de resultados",
        "apresentacao de resultados"
    ]
}


In [ ]:
def find_skills_in_text(
    text: str,
    skill_aliases: dict[str, list[str]]
) -> list[str]:
    """
    Identifica competências com base em aliases conhecidos.
    """

    normalized_text = text.lower()
    found_skills = []

    for skill, aliases in skill_aliases.items():
        for alias in aliases:
            if alias.lower() in normalized_text:
                found_skills.append(skill)
                break

    return found_skills


In [ ]:
job_skills = find_skills_in_text(
    job_description,
    skill_aliases
)

print("Competências identificadas na vaga:")
for skill in job_skills:
    print(f"- {skill}")


Competências identificadas na vaga:
- Python
- Machine Learning
- Análise de dados
- Modelos de linguagem
- Processamento de Linguagem Natural
- Avaliação de modelos
- Pesquisa
- Comunicação


In [ ]:
def build_skill_evidence_index(
    chunks: list[dict],
    skill_aliases: dict[str, list[str]]
) -> dict[str, list[dict]]:
    """
    Cria um índice que relaciona competências aos chunks que as mencionam.
    """

    evidence_index = {
        skill: []
        for skill in skill_aliases
    }

    for chunk in chunks:
        found_skills = find_skills_in_text(
            chunk["content"],
            skill_aliases
        )

        for skill in found_skills:
            evidence_index[skill].append({
                "chunk_id": chunk["chunk_id"],
                "source_name": chunk["source_name"],
                "source_type": chunk["source_type"],
                "page": chunk["metadata"].get("page"),
                "content": chunk["content"]
            })

    return evidence_index


In [ ]:
skill_evidence_index = build_skill_evidence_index(
    chunks=chunks,
    skill_aliases=skill_aliases
)


In [ ]:
for skill, evidences in skill_evidence_index.items():
    if evidences:
        print(f"\n{skill}: {len(evidences)} evidência(s)")

        for evidence in evidences[:2]:
            print(f"- Fonte: {evidence['source_name']}")
            print(f"- Página: {evidence['page']}")
            print(f"- Trecho: {evidence['content'][:250]}")



Python: 3 evidência(s)
- Fonte: 02_relatorio_projeto_ficticio.pdf
- Página: 1
- Trecho: Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks
acadêmicos por tema e prioridade
- Fonte: 01_curriculo_candidato_ficticio.pdf
- Página: 1
- Trecho: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.

Machine Learning: 1 evidência(s)
- Fonte: 01_curriculo_candidato_ficticio.pdf
- Página: 2
- Trecho: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre

Análise de d

In [ ]:
def analyze_skills_and_gaps(
    job_description: str,
    chunks: list[dict],
    skill_aliases: dict[str, list[str]]
) -> dict:
    """
    Analisa competências da vaga e verifica evidências no perfil.
    """

    job_skills = find_skills_in_text(
        job_description,
        skill_aliases
    )

    evidence_index = build_skill_evidence_index(
        chunks,
        skill_aliases
    )

    found_skills = []
    missing_skills = []
    skill_analysis = []

    for skill in job_skills:
        evidences = evidence_index.get(skill, [])

        if evidences:
            status = "found"
            found_skills.append(skill)
        else:
            status = "not_found"
            missing_skills.append(skill)

        skill_analysis.append({
            "skill": skill,
            "status": status,
            "evidence_count": len(evidences),
            "evidence": evidences[:3]
        })

    return {
        "job_skills": job_skills,
        "found_skills": found_skills,
        "missing_skills": missing_skills,
        "skill_analysis": skill_analysis
    }


In [ ]:
skills_analysis = analyze_skills_and_gaps(
    job_description=job_description,
    chunks=chunks,
    skill_aliases=skill_aliases
)


In [ ]:
print("COMPETÊNCIAS ENCONTRADAS")
print("=" * 60)

for skill in skills_analysis["found_skills"]:
    print(f"- {skill}")

print("\nCOMPETÊNCIAS NÃO ENCONTRADAS")
print("=" * 60)

for skill in skills_analysis["missing_skills"]:
    print(f"- {skill}")



COMPETÊNCIAS ENCONTRADAS
- Python
- Machine Learning
- Análise de dados
- Modelos de linguagem
- Processamento de Linguagem Natural
- Avaliação de modelos
- Pesquisa

COMPETÊNCIAS NÃO ENCONTRADAS
- Comunicação


In [ ]:
def calculate_skill_coverage(
    found_skills: list[str],
    job_skills: list[str]
) -> float:
    """
    Calcula a proporção de competências da vaga com evidências.
    """

    if not job_skills:
        return 0.0

    return len(found_skills) / len(job_skills)


In [ ]:
coverage = calculate_skill_coverage(
    found_skills=skills_analysis["found_skills"],
    job_skills=skills_analysis["job_skills"]
)

print(f"Cobertura de competências: {coverage:.2%}")


Cobertura de competências: 87.50%


In [ ]:
def retrieve_skill_evidence(
    skill: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 3,
    minimum_score: float = 0.35
) -> list[dict]:
    """
    Recupera evidências semanticamente relacionadas a uma competência.
    """

    results = semantic_search(
        query=skill,
        chunks=chunks,
        chunk_embeddings=chunk_embeddings,
        model=model,
        top_k=top_k
    )

    return [
        result
        for result in results
        if result["similarity_score"] >= minimum_score
    ]


In [ ]:
python_evidence = retrieve_skill_evidence(
    skill="Análise de dados",
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=3
)

for evidence in python_evidence:
    print(f"Score: {evidence['similarity_score']:.4f}")
    print(f"Fonte: {evidence['source_name']}")
    print(f"Conteúdo: {evidence['content'][:400]}")
    print("-" * 80)


Score: 0.5523
Fonte: Experiência acadêmica
Conteúdo: Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com
 uma base de dados extensa, análise de resultados e comparação de métricas.
--------------------------------------------------------------------------------
Score: 0.5411
Fonte: 01_curriculo_candidato_ficticio.pdf
Conteúdo: Currículo de Demonstração
Marina Oliveira
Candidata fictícia para testes do Pathfolio
Objetivo
Buscar oportunidade de estágio em Inteligência Artificial, Dados ou Desenvolvimento de Aplicações
Inteligentes.
Formação
Graduação em Sistemas de Informação
Universidade Exemplo — conclusão prevista: 2027
Experiência acadêmica
Projeto de iniciação científica — Avaliação de modelos de linguagem

Comparaç
--------------------------------------------------------------------------------
Score: 0.4900
Fonte: 02_relatorio_projeto_ficticio.pdf
Conteúdo: dados, métricas de avaliaç

In [ ]:
def analyze_skill_hybrid(
    skill: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    skill_aliases: dict[str, list[str]],
    minimum_score: float = 0.35
) -> dict:
    """
    Analisa uma competência com evidência direta e semântica.

    Evidência direta confirma a competência.
    Evidência semântica isolada é marcada como candidata.
    """

    direct_evidence = []

    for chunk in chunks:
        found_skills = find_skills_in_text(
            chunk["content"],
            {skill: skill_aliases.get(skill, [])}
        )

        if skill in found_skills:
            direct_evidence.append({
                "chunk_id": chunk["chunk_id"],
                "source_name": chunk["source_name"],
                "source_type": chunk["source_type"],
                "page": chunk["metadata"].get("page"),
                "content": chunk["content"],
                "evidence_method": "keyword"
            })

    semantic_evidence = retrieve_skill_evidence(
        skill=skill,
        chunks=chunks,
        chunk_embeddings=chunk_embeddings,
        model=model,
        top_k=3,
        minimum_score=minimum_score
    )

    semantic_only_evidence = []
    direct_ids = {
        evidence["chunk_id"]
        for evidence in direct_evidence
    }

    for evidence in semantic_evidence:
        if evidence["chunk_id"] not in direct_ids:
            evidence_copy = evidence.copy()
            evidence_copy["evidence_method"] = "semantic_candidate"
            semantic_only_evidence.append(evidence_copy)

    if direct_evidence:
        status = "found"
        evidence_level = "direct"
    elif semantic_only_evidence:
        status = "candidate"
        evidence_level = "semantic_only"
    else:
        status = "not_found"
        evidence_level = "none"

    return {
        "skill": skill,
        "status": status,
        "evidence_level": evidence_level,
        "direct_evidence": direct_evidence[:5],
        "semantic_candidates": semantic_only_evidence[:5]
    }


In [ ]:
def analyze_job_hybrid(
    job_description: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    skill_aliases: dict[str, list[str]],
    minimum_score: float = 0.35
) -> dict:
    """
    Analisa competências da vaga usando evidências diretas e semânticas.
    """

    job_skills = find_skills_in_text(
        job_description,
        skill_aliases
    )

    skill_results = []

    for skill in job_skills:
        result = analyze_skill_hybrid(
            skill=skill,
            chunks=chunks,
            chunk_embeddings=chunk_embeddings,
            model=model,
            skill_aliases=skill_aliases,
            minimum_score=minimum_score
        )

        skill_results.append(result)

    found_skills = [
        result["skill"]
        for result in skill_results
        if result["status"] == "found"
    ]

    candidate_skills = [
        result["skill"]
        for result in skill_results
        if result["status"] == "candidate"
    ]

    missing_skills = [
        result["skill"]
        for result in skill_results
        if result["status"] == "not_found"
    ]

    return {
        "job_skills": job_skills,
        "found_skills": found_skills,
        "candidate_skills": candidate_skills,
        "missing_skills": missing_skills,
        "skill_results": skill_results,
        "coverage": calculate_skill_coverage(
            found_skills,
            job_skills
        )
    }


In [ ]:
hybrid_analysis = analyze_job_hybrid(
    job_description=job_description,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    skill_aliases=skill_aliases,
    minimum_score=0.35
)


In [ ]:
print(hybrid_analysis.keys())


dict_keys(['job_skills', 'found_skills', 'candidate_skills', 'missing_skills', 'skill_results', 'coverage'])


In [ ]:
print(f"Cobertura: {hybrid_analysis['coverage']:.2%}")

print("\nEncontradas:")
for skill in hybrid_analysis["found_skills"]:
    print(f"- {skill}")

print("\nCandidatas:")
for skill in hybrid_analysis["candidate_skills"]:
    print(f"- {skill}")

print("\nNão encontradas:")
for skill in hybrid_analysis["missing_skills"]:
    print(f"- {skill}")


Cobertura: 87.50%

Encontradas:
- Python
- Machine Learning
- Análise de dados
- Modelos de linguagem
- Processamento de Linguagem Natural
- Avaliação de modelos
- Pesquisa

Candidatas:
- Comunicação

Não encontradas:


In [ ]:
for result in hybrid_analysis["skill_results"]:
    print(f"Competência: {result['skill']}")
    print(f"Status: {result['status']}")
    print(f"Nível de evidência: {result['evidence_level']}")

    print("\nEvidências diretas:")
    for evidence in result["direct_evidence"]:
        print(f"- Fonte: {evidence['source_name']}")
        print(f"- Página: {evidence['page']}")
        print(f"- Trecho: {evidence['content'][:500]}")

    print("\nCandidatas semânticas:")
    for evidence in result["semantic_candidates"]:
        print(f"- Score: {evidence['similarity_score']:.4f}")
        print(f"- Fonte: {evidence['source_name']}")
        print(f"- Trecho: {evidence['content'][:500]}")

    print("=" * 100)


Competência: Python
Status: found
Nível de evidência: direct

Evidências diretas:
- Fonte: 02_relatorio_projeto_ficticio.pdf
- Página: 1
- Trecho: Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks
acadêmicos por tema e prioridade, facilitando a análise por professores.
Atividades realizadas

Coleta e limpeza de exemplos textuais fictícios.

Definição de categorias para classificação dos feedbacks.

Criação de uma rotina de pré-processamento em Python.

Avaliação qualitat
- Fonte: 01_curriculo_candidato_ficticio.pdf
- Página: 1
- Trecho: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
- Fonte: 01_curriculo_candidato_ficticio.pdf
- Página: 2


In [ ]:
for result in hybrid_analysis["skill_results"]:
    print(
        f"{result['skill']} | "
        f"status={result['status']} | "
        f"evidência={result['evidence_level']} | "
        f"diretas={len(result['direct_evidence'])} | "
        f"candidatas={len(result['semantic_candidates'])}"
    )


Python | status=found | evidência=direct | diretas=3 | candidatas=0
Machine Learning | status=found | evidência=direct | diretas=1 | candidatas=2
Análise de dados | status=found | evidência=direct | diretas=1 | candidatas=3
Modelos de linguagem | status=found | evidência=direct | diretas=3 | candidatas=1
Processamento de Linguagem Natural | status=found | evidência=direct | diretas=4 | candidatas=0
Avaliação de modelos | status=found | evidência=direct | diretas=2 | candidatas=2
Pesquisa | status=found | evidência=direct | diretas=2 | candidatas=2
Comunicação | status=candidate | evidência=semantic_only | diretas=0 | candidatas=1


In [ ]:
def show_missing_skills(analysis: dict) -> None:
    missing_skills = analysis["missing_skills"]

    if not missing_skills:
        print("Nenhuma lacuna foi identificada.")
        return

    print("Competências sem evidência direta:")
    for skill in missing_skills:
        print(f"- {skill}")


In [ ]:
show_missing_skills(hybrid_analysis)


Nenhuma lacuna foi identificada.


In [ ]:
def build_skill_report(analysis: dict) -> dict:
    return {
        "coverage": analysis["coverage"],
        "job_skills": analysis["job_skills"],
        "found_skills": analysis["found_skills"],
        "candidate_skills": analysis["candidate_skills"],
        "missing_skills": analysis["missing_skills"],
        "summary": {
            "total_skills": len(analysis["job_skills"]),
            "found_count": len(analysis["found_skills"]),
            "candidate_count": len(analysis["candidate_skills"]),
            "missing_count": len(analysis["missing_skills"])
        }
    }


skill_report = build_skill_report(hybrid_analysis)
skill_report


{'coverage': 0.875,
 'job_skills': ['Python',
  'Machine Learning',
  'Análise de dados',
  'Modelos de linguagem',
  'Processamento de Linguagem Natural',
  'Avaliação de modelos',
  'Pesquisa',
  'Comunicação'],
 'found_skills': ['Python',
  'Machine Learning',
  'Análise de dados',
  'Modelos de linguagem',
  'Processamento de Linguagem Natural',
  'Avaliação de modelos',
  'Pesquisa'],
 'candidate_skills': ['Comunicação'],
 'missing_skills': [],
 'summary': {'total_skills': 8,
  'found_count': 7,
  'candidate_count': 1,
  'missing_count': 0}}

In [ ]:
import json

with open("pathfolio_skill_report.json", "w", encoding="utf-8") as file:
    json.dump(
        skill_report,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Relatório salvo.")


Relatório salvo.


In [ ]:
all_analyzed_skills = (
    hybrid_analysis["found_skills"]
    + hybrid_analysis["candidate_skills"]
    + hybrid_analysis["missing_skills"]
)

assert set(all_analyzed_skills) == set(
    hybrid_analysis["job_skills"]
)

assert not (
    set(hybrid_analysis["found_skills"])
    & set(hybrid_analysis["missing_skills"])
)

assert not (
    set(hybrid_analysis["candidate_skills"])
    & set(hybrid_analysis["missing_skills"])
)

print("Teste de consistência da análise aprovado.")


Teste de consistência da análise aprovado.


## 6. Geração de texto com LLM

Nesta etapa, o Pathfolio utilizará um modelo generativo para transformar
as evidências recuperadas em uma análise estruturada.

O modelo generativo será acessado pela API do Google Gemini, utilizando
uma chave armazenada com segurança nos Secrets do Google Colab.

A chave nunca será escrita diretamente no notebook, exibida na saída ou
enviada ao GitHub.

O modelo específico utilizado será definido na variável
`gemini_model_name`, pois sua disponibilidade pode variar conforme a
conta e a API utilizada.


In [99]:
!pip -q install -U google-genai


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.


In [100]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY2")

if not GEMINI_API_KEY:
    raise ValueError(
        "O Secret GEMINI_API_KEY não foi encontrado no Colab."
    )

print("Chave do Gemini encontrada com segurança.")


Chave do Gemini encontrada com segurança.


In [101]:
from google import genai

gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

gemini_model_name = "gemini-3.8-flash"

print(f"Modelo configurado: {gemini_model_name}")


Modelo configurado: gemini-3.8-flash


In [103]:
import time

def generate_with_retry(
    client,
    model,
    contents,
    attempts=3,
    wait_seconds=20
):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            print(f"Tentativa {attempt}/{attempts}...")

            response = client.models.generate_content(
                model=model,
                contents=contents
            )

            print("Resposta recebida.")
            return response

        except Exception as error:
            last_error = error
            print(f"Falhou: {type(error).__name__}")

            if attempt < attempts:
                print(f"Aguardando {wait_seconds} segundos...")
                time.sleep(wait_seconds)
                wait_seconds *= 2

    raise last_error


In [104]:
test_response = generate_with_retry(
    client=gemini_client,
    model=gemini_model_name,
    contents=(
        "Explique em uma frase, em português, o que é um currículo "
        "baseado em evidências."
    )
)

print(test_response.text)


Tentativa 1/3...
Falhou: ServerError
Aguardando 20 segundos...
Tentativa 2/3...
Resposta recebida.
Um currículo baseado em evidências é um plano educacional estruturado a partir de métodos e práticas pedagógicas cientificamente comprovados como eficazes para a aprendizagem dos alunos.


In [106]:
import time


def generate_with_retry(
    client,
    model,
    contents,
    config=None,
    attempts=3,
    wait_seconds=20
):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            print(f"Tentativa {attempt}/{attempts}...")

            request_args = {
                "model": model,
                "contents": contents
            }

            if config is not None:
                request_args["config"] = config

            response = client.models.generate_content(
                **request_args
            )

            print("Resposta recebida.")
            return response

        except Exception as error:
            last_error = error
            print(f"Falhou: {type(error).__name__}")

            if attempt < attempts:
                print(f"Aguardando {wait_seconds} segundos...")
                time.sleep(wait_seconds)
                wait_seconds *= 2

    raise last_error


In [107]:
def prepare_evidence_context(
    analysis: dict,
    max_evidence_per_skill: int = 2
) -> str:
    sections = []

    for result in analysis["skill_results"]:
        skill = result["skill"]
        status = result["status"]
        evidence_level = result["evidence_level"]

        sections.append(f"Competência: {skill}")
        sections.append(f"Status: {status}")
        sections.append(f"Nível da evidência: {evidence_level}")

        direct_evidence = result.get("direct_evidence", [])

        if direct_evidence:
            sections.append("Evidências diretas:")

            for evidence in direct_evidence[:max_evidence_per_skill]:
                sections.append(
                    f"ID: {evidence['chunk_id']}\n"
                    f"Fonte: {evidence['source_name']}\n"
                    f"Página: {evidence.get('page')}\n"
                    f"Trecho: {evidence['content']}"
                )

        semantic_candidates = result.get(
            "semantic_candidates",
            []
        )

        if semantic_candidates:
            sections.append("Evidências candidatas semânticas:")

            for evidence in semantic_candidates[:max_evidence_per_skill]:
                sections.append(
                    f"ID candidato: {evidence['chunk_id']}\n"
                    f"Fonte: {evidence['source_name']}\n"
                    f"Página: {evidence.get('page')}\n"
                    f"Trecho candidato: {evidence['content']}"
                )

        sections.append("-" * 60)

    return "\n".join(sections)


In [108]:
evidence_context = prepare_evidence_context(
    hybrid_analysis
)

print(evidence_context)


Competência: Python
Status: found
Nível da evidência: direct
Evidências diretas:
ID: chunk-0003
Fonte: 02_relatorio_projeto_ficticio.pdf
Página: 1
Trecho: Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks
acadêmicos por tema e prioridade, facilitando a análise por professores.
Atividades realizadas

Coleta e limpeza de exemplos textuais fictícios.

Definição de categorias para classificação dos feedbacks.

Criação de uma rotina de pré-processamento em Python.

Avaliação qualitativa dos resultados e análise de erros.

Registro das decisões e limitações do experimento.
Tecnologias e conceitos
Python, Pandas, classificação de texto, análise exploratória de dados, métricas de avaliação e Git.
Resultado
O experimento identificou situações em que textos ambíguos exigiam revisão
ID: chunk-0006
Fonte: 01_curriculo_candidato

In [109]:
analysis_schema = {
    "type": "object",
    "properties": {
        "professional_summary": {
            "type": "string",
            "description": (
                "Resumo profissional curto, baseado somente nas evidências."
            )
        },
        "strengths": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "skill": {
                        "type": "string"
                    },
                    "evidence_ids": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "explanation": {
                        "type": "string"
                    }
                },
                "required": [
                    "skill",
                    "evidence_ids",
                    "explanation"
                ]
            }
        },
        "gaps": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "skill": {
                        "type": "string"
                    },
                    "explanation": {
                        "type": "string"
                    }
                },
                "required": [
                    "skill",
                    "explanation"
                ]
            }
        },
        "warnings": {
            "type": "array",
            "items": {
                "type": "string"
            }
        }
    },
    "required": [
        "professional_summary",
        "strengths",
        "gaps",
        "warnings"
    ]
}


In [110]:
structured_prompt = f"""
Você é um assistente de carreira responsável por analisar
um candidato para uma vaga profissional.

O objetivo é analisar um currículo profissional.
Não confunda currículo profissional com currículo escolar
ou plano educacional.

Regras obrigatórias:

1. Use somente as evidências fornecidas.
2. Não invente experiências, cargos, datas, instituições ou habilidades.
3. Não transforme uma evidência candidata em uma competência confirmada.
4. Se não houver evidência suficiente, registre a informação em gaps.
5. Preserve o significado das experiências originais.
6. Responda em português.
7. Retorne somente o objeto JSON solicitado.
8. Use os IDs reais das evidências quando fizer uma associação.
9. Não crie IDs novos.

Descrição da vaga:
{job_description}

Análise de competências:
{evidence_context}
"""


In [111]:
structured_response = generate_with_retry(
    client=gemini_client,
    model=gemini_model_name,
    contents=structured_prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": analysis_schema
    }
)


Tentativa 1/3...
Resposta recebida.


In [112]:
print(structured_response.text)


{"professional_summary":"Marina Oliveira é estudante de graduação em Sistemas de Informação com previsão de conclusão em 2027. Possui experiência acadêmica em projetos de iniciação científica e projetos práticos com foco em inteligência artificial, processamento de linguagem natural, avaliação e uso de modelos de linguagem, manipulação e análise de dados com Python.","strengths":[{"skill":"Python","evidence_ids":["chunk-0003","chunk-0006"],"explanation":"Utilizou Python na criação de rotinas de pré-processamento de texto em projeto de classificação de feedbacks e no desenvolvimento de assistente experimental de estudos."},{"skill":"Machine Learning","evidence_ids":["chunk-0007"],"explanation":"Consta no currículo realização de curso livre introdutório em Machine Learning e listagem da competência entre suas habilidades."},{"skill":"Análise de dados","evidence_ids":["chunk-0007"],"explanation":"Apresenta formação em curso livre de Python para análise de dados e lista análise de dados en

In [113]:
import json

structured_analysis = json.loads(
    structured_response.text
)

print(structured_analysis.keys())


dict_keys(['professional_summary', 'strengths', 'gaps', 'warnings'])


In [114]:
assert "professional_summary" in structured_analysis
assert "strengths" in structured_analysis
assert "gaps" in structured_analysis
assert "warnings" in structured_analysis

assert isinstance(
    structured_analysis["professional_summary"],
    str
)

assert isinstance(
    structured_analysis["strengths"],
    list
)

assert isinstance(
    structured_analysis["gaps"],
    list
)

assert isinstance(
    structured_analysis["warnings"],
    list
)

print("Resposta estruturada validada.")


Resposta estruturada validada.


In [115]:
valid_chunk_ids = {
    chunk["chunk_id"]
    for chunk in chunks
}

used_evidence_ids = set()

for strength in structured_analysis["strengths"]:
    for evidence_id in strength["evidence_ids"]:
        used_evidence_ids.add(evidence_id)

invalid_ids = used_evidence_ids - valid_chunk_ids

if invalid_ids:
    print("IDs inválidos encontrados:", invalid_ids)
else:
    print("Todos os IDs de evidência são válidos.")


Todos os IDs de evidência são válidos.


In [116]:
print("RESUMO PROFISSIONAL")
print("=" * 70)
print(structured_analysis["professional_summary"])

print("\nPONTOS FORTES")
print("=" * 70)

for strength in structured_analysis["strengths"]:
    print(f"- {strength['skill']}")
    print(f"  Evidências: {strength['evidence_ids']}")
    print(f"  Explicação: {strength['explanation']}")

print("\nLACUNAS")
print("=" * 70)

for gap in structured_analysis["gaps"]:
    print(f"- {gap['skill']}")
    print(f"  {gap['explanation']}")

print("\nAVISOS")
print("=" * 70)

for warning in structured_analysis["warnings"]:
    print(f"- {warning}")


RESUMO PROFISSIONAL
Marina Oliveira é estudante de graduação em Sistemas de Informação com previsão de conclusão em 2027. Possui experiência acadêmica em projetos de iniciação científica e projetos práticos com foco em inteligência artificial, processamento de linguagem natural, avaliação e uso de modelos de linguagem, manipulação e análise de dados com Python.

PONTOS FORTES
- Python
  Evidências: ['chunk-0003', 'chunk-0006']
  Explicação: Utilizou Python na criação de rotinas de pré-processamento de texto em projeto de classificação de feedbacks e no desenvolvimento de assistente experimental de estudos.
- Machine Learning
  Evidências: ['chunk-0007']
  Explicação: Consta no currículo realização de curso livre introdutório em Machine Learning e listagem da competência entre suas habilidades.
- Análise de dados
  Evidências: ['chunk-0007']
  Explicação: Apresenta formação em curso livre de Python para análise de dados e lista análise de dados entre suas competências técnicas.
- Modelo

In [117]:
resume_schema = {
    "type": "object",
    "properties": {
        "professional_summary": {
            "type": "string",
            "description": (
                "Resumo profissional curto e fiel às evidências."
            )
        },
        "adapted_experiences": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "title": {
                        "type": "string"
                    },
                    "description": {
                        "type": "string"
                    },
                    "evidence_ids": {
                        "type": "array",
                        "items": {
                            "type": "string"
                        }
                    },
                    "evidence_type": {
                        "type": "string",
                        "enum": [
                            "practical_experience",
                            "academic_experience",
                            "project",
                            "course_or_training",
                            "declared_skill"
                        ]
                    }
                },
                "required": [
                    "title",
                    "description",
                    "evidence_ids",
                    "evidence_type"
                ]
            }
        },
        "missing_information": {
            "type": "array",
            "items": {
                "type": "string"
            }
        },
        "warnings": {
            "type": "array",
            "items": {
                "type": "string"
            }
        }
    },
    "required": [
        "professional_summary",
        "adapted_experiences",
        "missing_information",
        "warnings"
    ]
}


In [118]:
resume_generation_prompt = f"""
Você é um assistente especializado em adaptar currículos profissionais.

A tarefa é criar uma versão inicial de currículo direcionada à vaga,
usando somente as evidências fornecidas.

Descrição da vaga:
{job_description}

Análise de competências:
{evidence_context}

Resumo da análise anterior:
{json.dumps(structured_analysis, ensure_ascii=False, indent=2)}

Regras obrigatórias:

1. Não invente experiências, cargos, empresas, datas, instituições ou tecnologias.
2. Não transforme um curso em experiência profissional.
3. Não transforme uma habilidade declarada em experiência prática.
4. Preserve o significado original das experiências.
5. Cada experiência deve utilizar somente IDs reais das evidências.
6. Se uma informação não estiver disponível, registre isso em missing_information.
7. Não inclua uma competência apenas porque aparece na descrição da vaga.
8. Escreva em português profissional e claro.
9. Não use primeira pessoa.
10. Não inclua informações pessoais que não estejam nas evidências.
11. Retorne somente JSON válido conforme o schema.

Classificação das evidências:

- practical_experience: experiência prática profissional comprovada;
- academic_experience: iniciação científica ou atividade acadêmica;
- project: projeto pessoal ou acadêmico;
- course_or_training: curso, treinamento ou formação;
- declared_skill: habilidade listada sem detalhes suficientes de aplicação.

Para cada item adaptado, escolha apenas uma dessas classificações.
"""


In [119]:
resume_response = generate_with_retry(
    client=gemini_client,
    model=gemini_model_name,
    contents=resume_generation_prompt,
    config={
        "response_mime_type": "application/json",
        "response_json_schema": resume_schema
    }
)


Tentativa 1/3...
Resposta recebida.


In [120]:
resume_draft = json.loads(
    resume_response.text
)

print(resume_draft.keys())


dict_keys(['professional_summary', 'adapted_experiences', 'missing_information', 'warnings'])


In [121]:
print("RESUMO PROFISSIONAL")
print("=" * 70)
print(resume_draft["professional_summary"])

print("\nEXPERIÊNCIAS ADAPTADAS")
print("=" * 70)

for experience in resume_draft["adapted_experiences"]:
    print(f"\nTítulo: {experience['title']}")
    print(f"Tipo: {experience['evidence_type']}")
    print(f"Evidências: {experience['evidence_ids']}")
    print(f"Descrição: {experience['description']}")

print("\nINFORMAÇÕES AUSENTES")
print("=" * 70)

for item in resume_draft["missing_information"]:
    print(f"- {item}")

print("\nAVISOS")
print("=" * 70)

for warning in resume_draft["warnings"]:
    print(f"- {warning}")


RESUMO PROFISSIONAL
Estudante de Sistemas de Informação com foco em Inteligência Artificial, Processamento de Linguagem Natural e Modelos de Linguagem. Possui experiência acadêmica em iniciação científica com comparação e avaliação de modelos de IA, além de projetos em Python para classificação de textos e desenvolvimento de ferramentas de suporte a estudos.

EXPERIÊNCIAS ADAPTADAS

Título: Projeto de Iniciação Científica — Avaliação de Modelos de Linguagem
Tipo: academic_experience
Evidências: ['chunk-0000', 'chunk-0005']
Descrição: Atuação na comparação de modelos de inteligência artificial em análise textual e correção de redações, abrangendo organização e tratamento de base de dados, análise de métricas experimentais e documentação metodológica.

Título: Projeto — Classificação de feedbacks acadêmicos
Tipo: project
Evidências: ['chunk-0003']
Descrição: Investigação de técnicas de processamento de linguagem natural para organização de feedbacks textuais, incluindo coleta e limpeza d

In [122]:
valid_chunk_ids = {
    chunk["chunk_id"]
    for chunk in chunks
}

used_resume_ids = set()

for experience in resume_draft["adapted_experiences"]:
    used_resume_ids.update(
        experience["evidence_ids"]
    )

invalid_resume_ids = (
    used_resume_ids - valid_chunk_ids
)

if invalid_resume_ids:
    print(
        "IDs inválidos encontrados:",
        invalid_resume_ids
    )
else:
    print("Todos os IDs do currículo são válidos.")


Todos os IDs do currículo são válidos.


In [123]:
valid_evidence_types = {
    "practical_experience",
    "academic_experience",
    "project",
    "course_or_training",
    "declared_skill"
}

for experience in resume_draft["adapted_experiences"]:
    assert experience["evidence_type"] in valid_evidence_types

print("Tipos de evidência válidos.")


Tipos de evidência válidos.


In [124]:
assert isinstance(
    resume_draft["professional_summary"],
    str
)

assert isinstance(
    resume_draft["adapted_experiences"],
    list
)

assert isinstance(
    resume_draft["missing_information"],
    list
)

assert isinstance(
    resume_draft["warnings"],
    list
)

for experience in resume_draft["adapted_experiences"]:
    assert experience["title"]
    assert experience["description"]
    assert isinstance(
        experience["evidence_ids"],
        list
    )

print("Rascunho de currículo validado.")


Rascunho de currículo validado.


## 7. Montagem do currículo estruturado

Nesta etapa, o resultado gerado pelo modelo será organizado em uma estrutura
final de currículo.

A organização será feita em Python, sem uma nova chamada ao LLM, para evitar
alterações desnecessárias nas informações já analisadas.


In [125]:
final_resume = {
    "candidate": {
        "name": None,
        "headline": None,
        "contact": {}
    },
    "professional_summary": resume_draft[
        "professional_summary"
    ],
    "experiences": [],
    "education": [],
    "projects": [],
    "courses_and_training": [],
    "skills": [],
    "gaps": [],
    "warnings": resume_draft[
        "warnings"
    ],
    "metadata": {
        "job_description_used": job_description,
        "model": gemini_model_name,
        "source_count": len(cleaned_sources),
        "chunk_count": len(chunks)
    }
}


In [126]:
for experience in resume_draft["adapted_experiences"]:
    evidence_type = experience["evidence_type"]

    item = {
        "title": experience["title"],
        "description": experience["description"],
        "evidence_ids": experience["evidence_ids"],
        "evidence_type": evidence_type
    }

    if evidence_type == "academic_experience":
        final_resume["experiences"].append(item)

    elif evidence_type == "practical_experience":
        final_resume["experiences"].append(item)

    elif evidence_type == "project":
        final_resume["projects"].append(item)

    elif evidence_type == "course_or_training":
        final_resume["courses_and_training"].append(item)

    elif evidence_type == "declared_skill":
        final_resume["skills"].append(item)


In [127]:
for gap in structured_analysis.get("gaps", []):
    final_resume["gaps"].append({
        "skill": gap["skill"],
        "explanation": gap["explanation"]
    })


In [128]:
for skill in hybrid_analysis.get("candidate_skills", []):
    final_resume["gaps"].append({
        "skill": skill,
        "explanation": (
            "Foi encontrada uma associação semântica, mas não uma "
            "evidência direta suficiente."
        )
    })


In [129]:
education_item = {
    "title": "Graduação em Sistemas de Informação",
    "institution": "Universidade Exemplo",
    "completion_forecast": "2027",
    "evidence_ids": ["chunk-0000"]
}

final_resume["education"].append(education_item)



In [131]:
print("RESUMO PROFISSIONAL")
print("=" * 70)
print(final_resume["professional_summary"])

print("\nEXPERIÊNCIAS")
print("=" * 70)

for item in final_resume["experiences"]:
    print(f"\n{item['title']}")
    print(item["description"])
    print(f"Evidências: {item['evidence_ids']}")

print("\nPROJETOS")
print("=" * 70)

for item in final_resume["projects"]:
    print(f"\n{item['title']}")
    print(item["description"])
    print(f"Evidências: {item['evidence_ids']}")

print("\nCURSOS E TREINAMENTOS")
print("=" * 70)

for item in final_resume["courses_and_training"]:
    print(f"\n{item['title']}")
    print(item["description"])

print("\nHABILIDADES DECLARADAS")
print("=" * 70)

for item in final_resume["skills"]:
    print(f"\n{item['title']}")
    print(item["description"])

print("\nLACUNAS")
print("=" * 70)

for gap in final_resume["gaps"]:
    print(f"- {gap['skill']}: {gap['explanation']}")

print("\nAVISOS")
print("=" * 70)

for warning in final_resume["warnings"]:
    print(f"- {warning}")


RESUMO PROFISSIONAL
Estudante de Sistemas de Informação com foco em Inteligência Artificial, Processamento de Linguagem Natural e Modelos de Linguagem. Possui experiência acadêmica em iniciação científica com comparação e avaliação de modelos de IA, além de projetos em Python para classificação de textos e desenvolvimento de ferramentas de suporte a estudos.

EXPERIÊNCIAS

Projeto de Iniciação Científica — Avaliação de Modelos de Linguagem
Atuação na comparação de modelos de inteligência artificial em análise textual e correção de redações, abrangendo organização e tratamento de base de dados, análise de métricas experimentais e documentação metodológica.
Evidências: ['chunk-0000', 'chunk-0005']

PROJETOS

Projeto — Classificação de feedbacks acadêmicos
Investigação de técnicas de processamento de linguagem natural para organização de feedbacks textuais, incluindo coleta e limpeza de dados, criação de rotinas de pré-processamento com Python e Pandas e avaliação qualitativa de erros.
Ev

In [132]:
valid_chunk_ids = {
    chunk["chunk_id"]
    for chunk in chunks
}

all_final_evidence_ids = set()

for section_name in [
    "experiences",
    "projects",
    "courses_and_training",
    "skills"
]:
    for item in final_resume[section_name]:
        all_final_evidence_ids.update(
            item.get("evidence_ids", [])
        )

for education in final_resume["education"]:
    all_final_evidence_ids.update(
        education.get("evidence_ids", [])
    )

invalid_final_ids = (
    all_final_evidence_ids - valid_chunk_ids
)

if invalid_final_ids:
    print(
        "IDs inválidos encontrados:",
        invalid_final_ids
    )
else:
    print("Todos os IDs do currículo final são válidos.")


Todos os IDs do currículo final são válidos.


In [133]:
required_sections = [
    "candidate",
    "professional_summary",
    "experiences",
    "education",
    "projects",
    "courses_and_training",
    "skills",
    "gaps",
    "warnings",
    "metadata"
]

for section in required_sections:
    assert section in final_resume

assert isinstance(
    final_resume["professional_summary"],
    str
)

assert isinstance(
    final_resume["experiences"],
    list
)

assert isinstance(
    final_resume["projects"],
    list
)

assert isinstance(
    final_resume["education"],
    list
)

assert isinstance(
    final_resume["gaps"],
    list
)

print("Estrutura final do currículo validada.")


Estrutura final do currículo validada.


In [134]:
final_resume_path = "pathfolio_final_resume_debug.json"

with open(
    final_resume_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_resume,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Currículo estruturado salvo em: {final_resume_path}")


Currículo estruturado salvo em: pathfolio_final_resume_debug.json
